# Detector de armas v4 — meta de 90% no teste

Este notebook continua a partir de `best_v3_selecionado.pt`, recria o
dataset unificado e compara dois candidatos:

- `YOLO11n` continuado a partir do modelo v3;
- `YOLO11s` treinado no mesmo dataset para buscar mais capacidade.

Depois, o melhor candidato passa por um refinamento em resolução maior.

O notebook só marca a meta como atingida quando, no conjunto de teste:

- `mAP50 >= 0.90`;
- precisão média `>= 0.90`;
- recall médio `>= 0.85`;
- nenhuma classe fica com `mAP50 < 0.80`.

A meta não é garantida. O resultado é aceito apenas pelas métricas.
A exportação TFLite não é executada neste notebook para não travar a
sessão depois do treinamento.

In [ ]:
!pip -q install ultralytics==8.4.67 kagglehub pycocotools pyyaml tqdm pandas opencv-python-headless onnx onnxruntime

In [ ]:
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any
from collections import Counter, defaultdict
from google.colab import files
from IPython.display import display, Image as IPImage
from pycocotools.coco import COCO
from tqdm.auto import tqdm

import hashlib
import json
import os
import random
import shutil
import subprocess
import zipfile
import yaml

import cv2
import kagglehub
import numpy as np
import pandas as pd
import torch
import ultralytics
from ultralytics import YOLO


print('Ultralytics:', ultralytics.__version__)
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('Ative GPU T4 no Colab.')

## 1. Configuração

In [ ]:
@dataclass(frozen=True)
class Configuracao:
    seed: int = 42
    workers: int = 2

    epocas_nano: int = 70
    epocas_small_cabeca: int = 15
    epocas_small_completo: int = 70
    epocas_refino: int = 18

    imgsz_base: int = 768
    imgsz_refino: int = 896

    paciencia_nano: int = 16
    paciencia_small: int = 18
    paciencia_refino: int = 8

    lr_nano: float = 0.00018
    lr_small_cabeca: float = 0.00100
    lr_small_completo: float = 0.00035
    lr_refino: float = 0.00006
    lrf: float = 0.03

    usar_kaggle_guns_knives: bool = True
    usar_coco_positivos: bool = True
    usar_coco_hard_negatives: bool = True
    usar_zip_reforco_manual: bool = False

    max_coco_positivos: int = 2200
    max_candidatos_hard_negative: int = 5000

    max_hard_negatives_treino: int = 2200
    max_hard_negatives_val: int = 300
    max_hard_negatives_teste: int = 300

    max_negativos_limpos_treino: int = 700
    max_negativos_limpos_val: int = 120
    max_negativos_limpos_teste: int = 120

    conf_mineracao: float = 0.08
    conf_validacao: float = 0.25
    iou_validacao: float = 0.50

    conf_teste_externo: float = 0.25
    imgsz_teste_externo: int = 960

    meta_map50: float = 0.90
    meta_precisao: float = 0.90
    meta_recall: float = 0.85
    meta_map50_classe: float = 0.80

    preferir_nano_se_diferenca: float = 0.01


CFG = Configuracao()

CLASSES = {
    0: "machete",
    1: "knife",
    2: "baseball_bat",
    3: "rifle",
    4: "gun",
}

NOME_PARA_ID = {
    nome: indice
    for indice, nome in CLASSES.items()
}

RAIZ = Path("/content/armas_v4_meta90")
ORIG = RAIZ / "dataset_original"
KAGGLE = RAIZ / "dataset_kaggle"
COCO_DIR = RAIZ / "coco"
REFORCO = RAIZ / "reforco_manual"
UNIFICADO = RAIZ / "dataset_unificado"
RESULTADOS = RAIZ / "resultados"
ENTREGA = RAIZ / "entrega"
PESO_INICIAL = RAIZ / "best_v3_selecionado.pt"

for pasta in (
    RAIZ,
    ORIG,
    KAGGLE,
    COCO_DIR,
    REFORCO,
    RESULTADOS,
    ENTREGA,
):
    pasta.mkdir(
        parents=True,
        exist_ok=True,
    )

random.seed(CFG.seed)
np.random.seed(CFG.seed)
torch.manual_seed(CFG.seed)

display(pd.DataFrame([asdict(CFG)]))

## 2. Enviar o peso inicial `best_v3_selecionado.pt`

In [ ]:
enviados = files.upload()

pts = [
    nome
    for nome in enviados
    if nome.lower().endswith(".pt")
]

if len(pts) != 1:
    raise RuntimeError(
        "Envie exatamente um arquivo .pt: "
        "best_v3_selecionado.pt"
    )

shutil.copy2(
    pts[0],
    PESO_INICIAL,
)

modelo_base = YOLO(
    str(PESO_INICIAL)
)

print(
    "Peso inicial:",
    PESO_INICIAL,
)
print(
    "Classes:",
    modelo_base.names,
)

classes_peso = {
    int(chave): valor
    for chave, valor
    in modelo_base.names.items()
}

if classes_peso != CLASSES:
    raise RuntimeError(
        "As classes do peso não correspondem "
        "às classes esperadas."
    )

## 3. Baixar Dangerous Items Dataset

In [ ]:
URL_DANGEROUS = 'https://zenodo.org/records/16422779/files/Dangerous%20Items.zip?download=1'
ZIP_DANGEROUS = RAIZ/'dangerous_items.zip'

if not ZIP_DANGEROUS.exists():
    subprocess.run(['wget','-c',URL_DANGEROUS,'-O',str(ZIP_DANGEROUS)], check=True)

if not (ORIG/'.extraido').exists():
    with zipfile.ZipFile(ZIP_DANGEROUS) as z:
        z.extractall(ORIG)
    (ORIG/'.extraido').touch()
print('Dangerous Items extraído em:', ORIG)

## 4. Baixar reforço Kaggle para gun/knife

In [ ]:
if CFG.usar_kaggle_guns_knives:
    try:
        caminho_kaggle = Path(kagglehub.dataset_download('iqmansingh/guns-knives-object-detection'))
        print('Guns-Knives:', caminho_kaggle)
    except Exception as e:
        caminho_kaggle = None
        print('Falha ao baixar Kaggle Guns-Knives. O treinamento continuará sem ele:', e)
else:
    caminho_kaggle = None
    print('Reforço Kaggle desativado.')

## 5. Baixar COCO val2017 para positivos e negativos difíceis

COCO é usado aqui de duas formas:

- positivos adicionais para `knife` e `baseball_bat`;
- negativos difíceis com pessoas, celulares, mochilas, roupas e objetos comuns.

Imagens que contêm `knife` ou `baseball bat` recebem rótulo positivo. Imagens sem essas classes, mas com pessoas/objetos comuns, entram com `.txt` vazio.

In [ ]:
import requests

def arquivo_zip_valido(caminho: Path) -> bool:
    return (
        caminho.exists()
        and caminho.stat().st_size > 0
        and zipfile.is_zipfile(caminho)
    )


def baixar_zip_com_fallback(
    destino: Path,
    urls: list[str],
) -> None:
    if arquivo_zip_valido(destino):
        print("ZIP já disponível:", destino)
        return

    if destino.exists():
        print("Removendo arquivo incompleto:", destino)
        destino.unlink()

    temporario = destino.with_suffix(
        destino.suffix + ".part"
    )

    if temporario.exists():
        temporario.unlink()

    ultimo_erro = None

    for url in urls:
        for verificar_ssl in (True, False):
            try:
                print(
                    "Baixando:",
                    url,
                    "| SSL:",
                    verificar_ssl,
                )

                with requests.get(
                    url,
                    stream=True,
                    allow_redirects=True,
                    timeout=(30, 180),
                    verify=verificar_ssl,
                    headers={
                        "User-Agent": (
                            "Mozilla/5.0 "
                            "violence-edge-ai-colab"
                        )
                    },
                ) as resposta:
                    resposta.raise_for_status()

                    total = int(
                        resposta.headers.get(
                            "content-length",
                            0,
                        )
                    )

                    with temporario.open("wb") as arquivo:
                        with tqdm(
                            total=total,
                            unit="B",
                            unit_scale=True,
                            desc=destino.name,
                        ) as barra:
                            for bloco in resposta.iter_content(
                                chunk_size=1024 * 1024
                            ):
                                if not bloco:
                                    continue

                                arquivo.write(bloco)
                                barra.update(len(bloco))

                temporario.replace(destino)

                if not arquivo_zip_valido(destino):
                    destino.unlink(missing_ok=True)
                    raise RuntimeError(
                        "O arquivo baixado não é um ZIP válido."
                    )

                print("Download concluído:", destino)
                return

            except Exception as erro:
                ultimo_erro = erro
                temporario.unlink(missing_ok=True)

                print(
                    "Tentativa falhou:",
                    type(erro).__name__,
                    erro,
                )

                if verificar_ssl:
                    print(
                        "Tentando novamente sem verificar "
                        "o certificado SSL."
                    )

    raise RuntimeError(
        "Não foi possível baixar o arquivo do COCO. "
        f"Último erro: {ultimo_erro}"
    )


if (
    CFG.usar_coco_positivos
    or CFG.usar_coco_hard_negatives
):
    COCO_IMG_ZIP = (
        COCO_DIR / "val2017.zip"
    )
    COCO_ANN_ZIP = (
        COCO_DIR
        / "annotations_trainval2017.zip"
    )

    baixar_zip_com_fallback(
        COCO_IMG_ZIP,
        [
            (
                "https://images.cocodataset.org/"
                "zips/val2017.zip"
            ),
            (
                "http://images.cocodataset.org/"
                "zips/val2017.zip"
            ),
        ],
    )

    baixar_zip_com_fallback(
        COCO_ANN_ZIP,
        [
            (
                "https://images.cocodataset.org/"
                "annotations/"
                "annotations_trainval2017.zip"
            ),
            (
                "http://images.cocodataset.org/"
                "annotations/"
                "annotations_trainval2017.zip"
            ),
        ],
    )

    pasta_imagens = (
        COCO_DIR / "val2017"
    )
    pasta_anotacoes = (
        COCO_DIR / "annotations"
    )

    if not pasta_imagens.exists():
        print("Extraindo imagens COCO...")

        with zipfile.ZipFile(
            COCO_IMG_ZIP
        ) as arquivo:
            arquivo.extractall(
                COCO_DIR
            )

    if not pasta_anotacoes.exists():
        print("Extraindo anotações COCO...")

        with zipfile.ZipFile(
            COCO_ANN_ZIP
        ) as arquivo:
            arquivo.extractall(
                COCO_DIR
            )

    anotacoes_esperadas = (
        pasta_anotacoes
        / "instances_val2017.json"
    )

    if not anotacoes_esperadas.exists():
        raise FileNotFoundError(
            "instances_val2017.json não foi encontrado "
            "depois da extração."
        )

    quantidade_imagens = len(
        list(
            pasta_imagens.glob("*.jpg")
        )
    )

    if quantidade_imagens < 5000:
        raise RuntimeError(
            "A pasta val2017 está incompleta: "
            f"{quantidade_imagens} imagens."
        )

    print("COCO pronto:", COCO_DIR)
    print(
        "Imagens val2017:",
        quantidade_imagens,
    )
    print(
        "Anotações:",
        anotacoes_esperadas,
    )

else:
    print("COCO desativado.")

## 6. Reforço manual opcional

Estrutura aceita:

```text
reforco.zip
├── images/train, images/val, images/test
└── labels/train, labels/val, labels/test
```

Negativos difíceis devem ter `.txt` vazio correspondente.

In [ ]:
if CFG.usar_zip_reforco_manual:
    enviados = files.upload()
    zips = [nome for nome in enviados if nome.lower().endswith('.zip')]
    if len(zips) != 1:
        raise RuntimeError('Envie exatamente um ZIP de reforço.')
    if REFORCO.exists():
        shutil.rmtree(REFORCO)
    REFORCO.mkdir(parents=True)
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall(REFORCO)
    print('Reforço manual extraído:', REFORCO)
else:
    print('Reforço manual desativado.')

## 7. Funções de organização e remapeamento YOLO

In [ ]:
EXT_IMG = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def recriar_dataset_unificado() -> None:
    if UNIFICADO.exists():
        shutil.rmtree(UNIFICADO)

    for split in ("train", "val", "test"):
        (UNIFICADO / "images" / split).mkdir(parents=True, exist_ok=True)
        (UNIFICADO / "labels" / split).mkdir(parents=True, exist_ok=True)


def imagens_em(pasta: Path | None) -> list[Path]:
    if pasta is None or not pasta.exists():
        return []

    return sorted(
        caminho
        for caminho in pasta.rglob("*")
        if caminho.is_file() and caminho.suffix.lower() in EXT_IMG
    )


def nome_unico(imagem: Path, prefixo: str) -> str:
    assinatura = hashlib.sha1(
        str(imagem.resolve()).encode("utf-8")
    ).hexdigest()[:12]
    nome_limpo = imagem.stem.replace(" ", "_").replace("/", "_")
    return f"{prefixo}_{assinatura}_{nome_limpo}"


def copiar_ou_linkar(origem: Path, destino: Path) -> None:
    destino.parent.mkdir(parents=True, exist_ok=True)

    if destino.exists() or destino.is_symlink():
        destino.unlink()

    try:
        destino.symlink_to(origem.resolve())
    except Exception:
        shutil.copy2(origem, destino)


def label_por_imagem(imagem: Path) -> Path:
    partes = list(imagem.parts)

    if "images" in partes:
        indice = len(partes) - 1 - partes[::-1].index("images")
        partes[indice] = "labels"
        return Path(*partes).with_suffix(".txt")

    return imagem.with_suffix(".txt")


def escrever_label(destino: Path, linhas: list[str]) -> None:
    texto = "\n".join(linhas)
    if linhas:
        texto += "\n"
    destino.write_text(texto, encoding="utf-8")


def copiar_dataset_yolo(
    imagens: list[Path],
    split: str,
    prefixo: str,
    mapa_classes: dict[int, int] | None = None,
) -> dict[str, Any]:
    objetos = 0
    negativas = 0
    sem_rotulo = 0

    for imagem in tqdm(imagens, desc=f"{prefixo}-{split}"):
        base = nome_unico(imagem, prefixo)
        destino_img = (
            UNIFICADO
            / "images"
            / split
            / f"{base}{imagem.suffix.lower()}"
        )
        destino_lbl = (
            UNIFICADO
            / "labels"
            / split
            / f"{base}.txt"
        )

        copiar_ou_linkar(imagem, destino_img)

        origem_lbl = label_por_imagem(imagem)
        linhas_saida = []

        if origem_lbl.exists():
            for linha in origem_lbl.read_text(
                encoding="utf-8",
                errors="ignore",
            ).splitlines():
                partes = linha.strip().split()

                if len(partes) != 5:
                    continue

                try:
                    classe_original = int(float(partes[0]))
                    coordenadas = [float(valor) for valor in partes[1:]]
                except ValueError:
                    continue

                nova_classe = (
                    mapa_classes.get(classe_original)
                    if mapa_classes is not None
                    else classe_original
                )

                if nova_classe not in CLASSES:
                    continue

                if any(valor < 0.0 or valor > 1.0 for valor in coordenadas):
                    continue

                linhas_saida.append(
                    " ".join([str(nova_classe), *partes[1:]])
                )
        else:
            sem_rotulo += 1

        escrever_label(destino_lbl, linhas_saida)
        objetos += len(linhas_saida)
        negativas += int(not linhas_saida)

    return {
        "fonte": prefixo,
        "split": split,
        "imagens": len(imagens),
        "objetos": objetos,
        "negativas": negativas,
        "sem_rotulo_origem": sem_rotulo,
    }


def normalizar_nome(nome: str) -> str:
    return (
        str(nome)
        .strip()
        .lower()
        .replace(" ", "_")
        .replace("-", "_")
    )


def nomes_para_dict(nomes: Any) -> dict[int, str]:
    if isinstance(nomes, list):
        return {indice: str(nome) for indice, nome in enumerate(nomes)}
    return {int(indice): str(nome) for indice, nome in nomes.items()}


def encontrar_yaml(pasta: Path):
    candidatos = [*pasta.rglob("*.yaml"), *pasta.rglob("*.yml")]

    for caminho in candidatos:
        try:
            dados = yaml.safe_load(caminho.read_text(encoding="utf-8"))

            if (
                isinstance(dados, dict)
                and {"train", "val", "names"} <= set(dados)
            ):
                return caminho, dados
        except Exception:
            continue

    return None, None


def resolver(valor: Any, base: Path) -> Path:
    caminho = Path(str(valor))
    if caminho.is_absolute():
        return caminho.resolve()
    return (base / caminho).resolve()


def encontrar_split(raiz: Path, nomes: tuple[str, ...]) -> Path | None:
    candidatos = []

    for nome in nomes:
        candidatos.extend(
            [
                raiz / nome / "images",
                raiz / "images" / nome,
                raiz / nome,
            ]
        )

    for candidato in candidatos:
        if imagens_em(candidato):
            return candidato

    for pasta in raiz.rglob("*"):
        if (
            pasta.is_dir()
            and pasta.name.lower() in {nome.lower() for nome in nomes}
        ):
            if imagens_em(pasta / "images"):
                return pasta / "images"
            if imagens_em(pasta):
                return pasta

    return None


recriar_dataset_unificado()
print("Dataset unificado recriado:", UNIFICADO)


## 8. Montar dataset unificado

In [ ]:
relatorios = []

raiz_danger = (
    ORIG / "Dangerous Items"
    if (ORIG / "Dangerous Items").exists()
    else ORIG
)

splits_danger = {
    "train": encontrar_split(raiz_danger, ("train", "training")),
    "val": encontrar_split(raiz_danger, ("valid", "val", "validation")),
    "test": encontrar_split(raiz_danger, ("test", "testing")),
}

if splits_danger["train"] is None or splits_danger["val"] is None:
    raise RuntimeError(
        f"Não encontrei train/val no Dangerous Items: {raiz_danger}"
    )

for split, pasta in splits_danger.items():
    if pasta is None:
        continue
    relatorios.append(
        copiar_dataset_yolo(
            imagens_em(pasta),
            split,
            "danger",
        )
    )

if caminho_kaggle is not None:
    yaml_gk, dados_gk = encontrar_yaml(caminho_kaggle)

    if yaml_gk is None:
        print(
            "Guns-Knives sem YAML reconhecido. "
            "Ignorado para evitar rótulos incorretos."
        )
    else:
        nomes = nomes_para_dict(dados_gk["names"])
        mapa = {}

        for indice, nome in nomes.items():
            nome_norm = normalizar_nome(nome)

            if nome_norm in {
                "gun",
                "guns",
                "pistol",
                "handgun",
                "firearm",
            }:
                mapa[indice] = NOME_PARA_ID["gun"]
            elif nome_norm in {"knife", "knives"}:
                mapa[indice] = NOME_PARA_ID["knife"]

        if mapa:
            base_gk = Path(dados_gk.get("path", "."))
            if not base_gk.is_absolute():
                base_gk = (yaml_gk.parent / base_gk).resolve()

            for split, chave in (
                ("train", "train"),
                ("val", "val"),
                ("test", "test"),
            ):
                if chave not in dados_gk:
                    continue

                imagens = imagens_em(
                    resolver(dados_gk[chave], base_gk)
                )

                if imagens:
                    relatorios.append(
                        copiar_dataset_yolo(
                            imagens,
                            split,
                            f"gk_{split}",
                            mapa,
                        )
                    )
        else:
            print("Classes gun/knife não encontradas:", nomes)

display(pd.DataFrame(relatorios))


## 9. Adicionar COCO: knife/baseball_bat + hard negatives

In [ ]:
if CFG.usar_coco_positivos or CFG.usar_coco_hard_negatives:
    ann = COCO_DIR / "annotations" / "instances_val2017.json"
    coco = COCO(str(ann))
    categorias = {
        item["name"]: item["id"]
        for item in coco.loadCats(coco.getCatIds())
    }

if CFG.usar_coco_positivos:
    mapa_positivo = {
        "knife": NOME_PARA_ID["knife"],
        "baseball bat": NOME_PARA_ID["baseball_bat"],
    }

    ids_cat = [
        categorias[nome]
        for nome in mapa_positivo
        if nome in categorias
    ]

    ids_imagens = set()

    for categoria_id in ids_cat:
        ids_imagens.update(
            coco.getImgIds(catIds=[categoria_id])
        )

    ids_imagens = list(ids_imagens)
    random.Random(CFG.seed).shuffle(ids_imagens)
    ids_imagens = ids_imagens[: CFG.max_coco_positivos]

    n = len(ids_imagens)
    corte_1 = int(n * 0.75)
    corte_2 = int(n * 0.875)

    divisao = {
        "train": ids_imagens[:corte_1],
        "val": ids_imagens[corte_1:corte_2],
        "test": ids_imagens[corte_2:],
    }

    for split, ids_split in divisao.items():
        objetos = 0

        for imagem_id in tqdm(
            ids_split,
            desc=f"COCO positivos {split}",
        ):
            info = coco.loadImgs([imagem_id])[0]
            origem = COCO_DIR / "val2017" / info["file_name"]
            base = f"coco_pos_{imagem_id}"
            destino_img = UNIFICADO / "images" / split / f"{base}.jpg"
            destino_lbl = UNIFICADO / "labels" / split / f"{base}.txt"

            copiar_ou_linkar(origem, destino_img)

            anotacoes = coco.loadAnns(
                coco.getAnnIds(
                    imgIds=[imagem_id],
                    catIds=ids_cat,
                    iscrowd=False,
                )
            )

            linhas = []

            for anotacao in anotacoes:
                nome_categoria = coco.loadCats(
                    [anotacao["category_id"]]
                )[0]["name"]

                if nome_categoria not in mapa_positivo:
                    continue

                x, y, largura, altura = anotacao["bbox"]

                if largura <= 0 or altura <= 0:
                    continue

                cx = (x + largura / 2) / info["width"]
                cy = (y + altura / 2) / info["height"]
                w = largura / info["width"]
                h = altura / info["height"]

                linhas.append(
                    (
                        f"{mapa_positivo[nome_categoria]} "
                        f"{cx:.6f} {cy:.6f} {w:.6f} {h:.6f}"
                    )
                )

            escrever_label(destino_lbl, linhas)
            objetos += len(linhas)

        relatorios.append(
            {
                "fonte": "coco_positivos",
                "split": split,
                "imagens": len(ids_split),
                "objetos": objetos,
                "negativas": 0,
                "sem_rotulo_origem": 0,
            }
        )

if CFG.usar_coco_hard_negatives:
    nomes_dificeis = [
        "person",
        "cell phone",
        "tie",
        "umbrella",
        "fork",
        "spoon",
        "scissors",
        "toothbrush",
        "bottle",
        "cup",
        "book",
        "backpack",
        "handbag",
    ]

    candidatos = set()

    for nome in nomes_dificeis:
        if nome in categorias:
            candidatos.update(
                coco.getImgIds(catIds=[categorias[nome]])
            )

    excluir = set()

    for nome in ("knife", "baseball bat"):
        if nome in categorias:
            excluir.update(
                coco.getImgIds(catIds=[categorias[nome]])
            )

    candidatos = list(candidatos - excluir)
    random.Random(CFG.seed + 10).shuffle(candidatos)
    candidatos = candidatos[: CFG.max_candidatos_hard_negative]

    pares = []

    for imagem_id in candidatos:
        info = coco.loadImgs([imagem_id])[0]
        pares.append(
            (
                imagem_id,
                COCO_DIR / "val2017" / info["file_name"],
            )
        )

    modelo_mineracao = YOLO(str(PESO_INICIAL))
    hard = []
    limpos = []
    tamanho_bloco = 64

    for inicio in tqdm(
        range(0, len(pares), tamanho_bloco),
        desc="Minerando hard negatives",
    ):
        bloco = pares[inicio : inicio + tamanho_bloco]

        resultados_bloco = modelo_mineracao.predict(
            source=[str(caminho) for _, caminho in bloco],
            imgsz=640,
            conf=CFG.conf_mineracao,
            iou=0.50,
            device=0,
            verbose=False,
        )

        for (imagem_id, caminho), resultado in zip(
            bloco,
            resultados_bloco,
            strict=True,
        ):
            if resultado.boxes is not None and len(resultado.boxes) > 0:
                maior = float(resultado.boxes.conf.max().cpu())
                hard.append((imagem_id, caminho, maior))
            else:
                limpos.append((imagem_id, caminho, 0.0))

    hard.sort(key=lambda item: item[2], reverse=True)
    random.Random(CFG.seed + 11).shuffle(limpos)

    limites_hard = {
        "train": CFG.max_hard_negatives_treino,
        "val": CFG.max_hard_negatives_val,
        "test": CFG.max_hard_negatives_teste,
    }
    limites_limpos = {
        "train": CFG.max_negativos_limpos_treino,
        "val": CFG.max_negativos_limpos_val,
        "test": CFG.max_negativos_limpos_teste,
    }

    def repartir(itens, limites):
        saida = {}
        cursor = 0
        for split in ("train", "val", "test"):
            quantidade = limites[split]
            saida[split] = itens[cursor : cursor + quantidade]
            cursor += quantidade
        return saida

    hard_split = repartir(hard, limites_hard)
    limpos_split = repartir(limpos, limites_limpos)

    for split in ("train", "val", "test"):
        itens = hard_split[split] + limpos_split[split]

        for imagem_id, origem, _ in itens:
            base = f"coco_neg_{split}_{imagem_id}"
            destino_img = UNIFICADO / "images" / split / f"{base}.jpg"
            destino_lbl = UNIFICADO / "labels" / split / f"{base}.txt"

            copiar_ou_linkar(origem, destino_img)
            escrever_label(destino_lbl, [])

        relatorios.append(
            {
                "fonte": "coco_hard_negatives",
                "split": split,
                "imagens": len(itens),
                "objetos": 0,
                "negativas": len(itens),
                "sem_rotulo_origem": 0,
            }
        )

    print("Hard negatives encontrados:", len(hard))

display(
    pd.DataFrame(relatorios)
    .groupby(["fonte", "split"])
    .sum(numeric_only=True)
    .reset_index()
)


## 10. Adicionar reforço manual, se existir

In [ ]:
if CFG.usar_zip_reforco_manual:
    raiz = REFORCO
    raiz_candidata = next((p.parent for p in raiz.rglob('images') if (p.parent/'labels').exists()), raiz)
    for split in ['train','val','test']:
        imgs = imagens_em(raiz_candidata/'images'/split)
        if imgs:
            relatorios.append(copiar_dataset_yolo(imgs, split, 'manual'))
else:
    print('Sem reforço manual.')

## 11. Auditar dataset unificado

In [ ]:
resumo = []
classes_count = Counter()

for split in ("train", "val", "test"):
    imagens = imagens_em(UNIFICADO / "images" / split)
    objetos = 0
    negativas = 0
    invalidos = 0

    for imagem in imagens:
        label = UNIFICADO / "labels" / split / f"{imagem.stem}.txt"

        if not label.exists():
            invalidos += 1
            continue

        linhas = [
            linha.strip()
            for linha in label.read_text(encoding="utf-8").splitlines()
            if linha.strip()
        ]

        negativas += int(not linhas)

        for linha in linhas:
            partes = linha.split()

            if len(partes) != 5:
                invalidos += 1
                continue

            try:
                classe_id = int(float(partes[0]))
                coordenadas = [float(valor) for valor in partes[1:]]
            except ValueError:
                invalidos += 1
                continue

            if (
                classe_id not in CLASSES
                or any(valor < 0.0 or valor > 1.0 for valor in coordenadas)
            ):
                invalidos += 1
                continue

            classes_count[(split, CLASSES[classe_id])] += 1
            objetos += 1

    resumo.append(
        {
            "split": split,
            "imagens": len(imagens),
            "objetos": objetos,
            "negativas": negativas,
            "invalidos": invalidos,
        }
    )

auditoria = pd.DataFrame(resumo)
display(auditoria)

tabela_classes = pd.DataFrame(
    [
        {
            "split": split,
            "classe": classe,
            "objetos": quantidade,
        }
        for (split, classe), quantidade in classes_count.items()
    ]
)

display(
    tabela_classes.pivot(
        index="classe",
        columns="split",
        values="objetos",
    )
    .fillna(0)
    .astype(int)
)

if auditoria["invalidos"].sum() > 0:
    raise RuntimeError("Há rótulos inválidos ou ausentes.")

minimos = {"train": 5000, "val": 900, "test": 900}

for split, minimo in minimos.items():
    quantidade = int(
        auditoria.loc[
            auditoria["split"] == split,
            "imagens",
        ].iloc[0]
    )
    if quantidade < minimo:
        raise RuntimeError(f"Poucas imagens em {split}: {quantidade}.")

for split in ("train", "val", "test"):
    for classe in CLASSES.values():
        if classes_count[(split, classe)] == 0:
            raise RuntimeError(f"Sem objetos de {classe} em {split}.")

yaml_v3 = RAIZ / "dados_armas_v3.yaml"
yaml_v3.write_text(
    yaml.safe_dump(
        {
            "path": str(UNIFICADO),
            "train": "images/train",
            "val": "images/val",
            "test": "images/test",
            "names": CLASSES,
        },
        allow_unicode=True,
        sort_keys=False,
    ),
    encoding="utf-8",
)

print(yaml_v3.read_text())

auditoria.to_csv(
    RESULTADOS / "auditoria_dataset_v3.csv",
    index=False,
)
pd.DataFrame(relatorios).to_csv(
    RESULTADOS / "fontes_dataset_v3.csv",
    index=False,
)


## 12. Treinar candidatos nano e small

In [ ]:
# ============================================================
# CANDIDATO 1 — YOLO11n continuado do best_v3
# ============================================================

modelo_nano = YOLO(
    str(PESO_INICIAL)
)

modelo_nano.train(
    data=str(yaml_v3),
    epochs=CFG.epocas_nano,
    imgsz=CFG.imgsz_base,
    batch=-1,
    patience=CFG.paciencia_nano,
    workers=CFG.workers,
    device=0,
    project=str(RESULTADOS),
    name="weapons_v4_nano",
    exist_ok=True,
    optimizer="AdamW",
    lr0=CFG.lr_nano,
    lrf=CFG.lrf,
    seed=CFG.seed,
    deterministic=True,
    amp=True,
    cos_lr=True,
    cache=False,
    mosaic=0.55,
    mixup=0.08,
    copy_paste=0.0,
    close_mosaic=12,
    translate=0.12,
    scale=0.45,
    fliplr=0.50,
    hsv_h=0.015,
    hsv_s=0.55,
    hsv_v=0.42,
    degrees=4.0,
    shear=2.0,
    perspective=0.0005,
    plots=True,
    save=True,
    save_period=10,
    verbose=True,
)

PASTA_NANO = (
    RESULTADOS
    / "weapons_v4_nano"
)
BEST_NANO = (
    PASTA_NANO
    / "weights"
    / "best.pt"
)

if not BEST_NANO.exists():
    raise FileNotFoundError(
        "best.pt do candidato nano "
        "não foi criado."
    )


# ============================================================
# CANDIDATO 2 — YOLO11s
# Primeiro treina a cabeça; depois libera o modelo completo.
# ============================================================

modelo_small_cabeca = YOLO(
    "yolo11s.pt"
)

modelo_small_cabeca.train(
    data=str(yaml_v3),
    epochs=CFG.epocas_small_cabeca,
    imgsz=CFG.imgsz_base,
    batch=-1,
    patience=CFG.epocas_small_cabeca,
    workers=CFG.workers,
    device=0,
    project=str(RESULTADOS),
    name="weapons_v4_small_cabeca",
    exist_ok=True,
    optimizer="AdamW",
    lr0=CFG.lr_small_cabeca,
    lrf=0.10,
    freeze=10,
    seed=CFG.seed + 10,
    deterministic=True,
    amp=True,
    cos_lr=True,
    cache=False,
    mosaic=0.65,
    mixup=0.10,
    close_mosaic=4,
    translate=0.12,
    scale=0.45,
    fliplr=0.50,
    hsv_h=0.015,
    hsv_s=0.55,
    hsv_v=0.42,
    plots=True,
    save=True,
    verbose=True,
)

BEST_SMALL_CABECA = (
    RESULTADOS
    / "weapons_v4_small_cabeca"
    / "weights"
    / "best.pt"
)

if not BEST_SMALL_CABECA.exists():
    raise FileNotFoundError(
        "best.pt da cabeça do small "
        "não foi criado."
    )

modelo_small = YOLO(
    str(BEST_SMALL_CABECA)
)

modelo_small.train(
    data=str(yaml_v3),
    epochs=CFG.epocas_small_completo,
    imgsz=CFG.imgsz_base,
    batch=-1,
    patience=CFG.paciencia_small,
    workers=CFG.workers,
    device=0,
    project=str(RESULTADOS),
    name="weapons_v4_small",
    exist_ok=True,
    optimizer="AdamW",
    lr0=CFG.lr_small_completo,
    lrf=CFG.lrf,
    freeze=0,
    seed=CFG.seed + 11,
    deterministic=True,
    amp=True,
    cos_lr=True,
    cache=False,
    mosaic=0.50,
    mixup=0.06,
    close_mosaic=12,
    translate=0.10,
    scale=0.40,
    fliplr=0.50,
    hsv_h=0.015,
    hsv_s=0.50,
    hsv_v=0.40,
    degrees=3.0,
    shear=1.5,
    perspective=0.0004,
    plots=True,
    save=True,
    save_period=10,
    verbose=True,
)

PASTA_SMALL = (
    RESULTADOS
    / "weapons_v4_small"
)
BEST_SMALL = (
    PASTA_SMALL
    / "weights"
    / "best.pt"
)

if not BEST_SMALL.exists():
    raise FileNotFoundError(
        "best.pt do candidato small "
        "não foi criado."
    )

print("Nano:", BEST_NANO)
print("Small:", BEST_SMALL)

## 13. Comparar, refinar e selecionar

In [ ]:
def avaliar_modelo(
    nome: str,
    caminho: Path,
    imgsz: int = 768,
):
    modelo = YOLO(
        str(caminho)
    )

    metricas = modelo.val(
        data=str(yaml_v3),
        split="test",
        imgsz=imgsz,
        batch=-1,
        conf=CFG.conf_validacao,
        iou=CFG.iou_validacao,
        device=0,
        project=str(RESULTADOS),
        name=f"validacao_{nome}",
        exist_ok=True,
        plots=True,
        verbose=True,
    )

    linha = {
        "modelo": nome,
        "mAP50-95": float(
            metricas.box.map
        ),
        "mAP50": float(
            metricas.box.map50
        ),
        "mAP75": float(
            metricas.box.map75
        ),
        "precisao_media": float(
            np.mean(
                metricas.box.p
            )
        ),
        "revocacao_media": float(
            np.mean(
                metricas.box.r
            )
        ),
    }

    por_classe = []

    for indice in range(
        len(metricas.box.maps)
    ):
        por_classe.append(
            {
                "modelo": nome,
                "classe": (
                    metricas.names[
                        indice
                    ]
                ),
                "precisao": float(
                    metricas.box.p[
                        indice
                    ]
                ),
                "revocacao": float(
                    metricas.box.r[
                        indice
                    ]
                ),
                "mAP50": float(
                    metricas.box.ap50[
                        indice
                    ]
                ),
                "mAP50-95": float(
                    metricas.box.maps[
                        indice
                    ]
                ),
            }
        )

    menor_map50 = min(
        item["mAP50"]
        for item in por_classe
    )

    penalidade = max(
        0.0,
        CFG.meta_map50_classe
        - menor_map50,
    )

    linha["menor_mAP50_classe"] = (
        menor_map50
    )

    linha["pontuacao"] = (
        0.40 * linha["mAP50"]
        + 0.25 * linha["mAP50-95"]
        + 0.20 * linha["precisao_media"]
        + 0.15 * linha["revocacao_media"]
        - 0.35 * penalidade
    )

    return linha, por_classe


candidatos_iniciais = {
    "v3_inicial": PESO_INICIAL,
    "v4_nano": BEST_NANO,
    "v4_small": BEST_SMALL,
}

linhas = []
classes_linhas = []

for nome, caminho in (
    candidatos_iniciais.items()
):
    linha, por_classe = (
        avaliar_modelo(
            nome,
            caminho,
            imgsz=CFG.imgsz_base,
        )
    )

    linhas.append(linha)
    classes_linhas.extend(
        por_classe
    )

comparacao_inicial = (
    pd.DataFrame(linhas)
    .sort_values(
        "pontuacao",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(comparacao_inicial)

melhor_nome = str(
    comparacao_inicial.iloc[0][
        "modelo"
    ]
)

melhor_peso = (
    candidatos_iniciais[
        melhor_nome
    ]
)

# Preferência pelo nano quando a diferença é mínima.
linha_nano = (
    comparacao_inicial[
        comparacao_inicial["modelo"]
        == "v4_nano"
    ]
)

linha_melhor = (
    comparacao_inicial.iloc[0]
)

if (
    not linha_nano.empty
    and melhor_nome == "v4_small"
):
    diferenca = float(
        linha_melhor["pontuacao"]
        - linha_nano.iloc[0][
            "pontuacao"
        ]
    )

    if (
        diferenca
        <= CFG.preferir_nano_se_diferenca
    ):
        melhor_nome = "v4_nano"
        melhor_peso = BEST_NANO

        print(
            "Nano escolhido por ter desempenho "
            "próximo e ser menor."
        )

print(
    "Candidato para refinamento:",
    melhor_nome,
    melhor_peso,
)


# ============================================================
# REFINAMENTO FINAL
# ============================================================

modelo_refino = YOLO(
    str(melhor_peso)
)

modelo_refino.train(
    data=str(yaml_v3),
    epochs=CFG.epocas_refino,
    imgsz=CFG.imgsz_refino,
    batch=-1,
    patience=CFG.paciencia_refino,
    workers=CFG.workers,
    device=0,
    project=str(RESULTADOS),
    name="weapons_v4_refino",
    exist_ok=True,
    optimizer="AdamW",
    lr0=CFG.lr_refino,
    lrf=0.10,
    seed=CFG.seed + 20,
    deterministic=True,
    amp=True,
    cos_lr=True,
    cache=False,
    mosaic=0.10,
    mixup=0.0,
    close_mosaic=3,
    translate=0.05,
    scale=0.18,
    fliplr=0.50,
    hsv_h=0.010,
    hsv_s=0.35,
    hsv_v=0.30,
    degrees=2.0,
    shear=1.0,
    plots=True,
    save=True,
    save_period=5,
    verbose=True,
)

BEST_REFINO = (
    RESULTADOS
    / "weapons_v4_refino"
    / "weights"
    / "best.pt"
)

if not BEST_REFINO.exists():
    raise FileNotFoundError(
        "best.pt do refinamento "
        "não foi criado."
    )

linha_refino, classes_refino = (
    avaliar_modelo(
        "v4_refino",
        BEST_REFINO,
        imgsz=CFG.imgsz_refino,
    )
)

linhas.append(linha_refino)
classes_linhas.extend(
    classes_refino
)

comparacao = (
    pd.DataFrame(linhas)
    .sort_values(
        "pontuacao",
        ascending=False,
    )
    .reset_index(drop=True)
)

por_classe_df = pd.DataFrame(
    classes_linhas
)

display(comparacao)
display(por_classe_df)

pesos_disponiveis = {
    "v3_inicial": PESO_INICIAL,
    "v4_nano": BEST_NANO,
    "v4_small": BEST_SMALL,
    "v4_refino": BEST_REFINO,
}

final_nome = str(
    comparacao.iloc[0][
        "modelo"
    ]
)

final_peso = (
    pesos_disponiveis[
        final_nome
    ]
)

linha_final = (
    comparacao.iloc[0]
)

classes_final = por_classe_df[
    por_classe_df["modelo"]
    == final_nome
]

atingiu_meta = bool(
    linha_final["mAP50"]
    >= CFG.meta_map50
    and linha_final[
        "precisao_media"
    ]
    >= CFG.meta_precisao
    and linha_final[
        "revocacao_media"
    ]
    >= CFG.meta_recall
    and classes_final[
        "mAP50"
    ].min()
    >= CFG.meta_map50_classe
)

BEST_FINAL = (
    RESULTADOS
    / "best_v4_selecionado.pt"
)

shutil.copy2(
    final_peso,
    BEST_FINAL,
)

comparacao.to_csv(
    RESULTADOS
    / "comparacao_modelos_v4.csv",
    index=False,
)

por_classe_df.to_csv(
    RESULTADOS
    / "metricas_por_classe_v4.csv",
    index=False,
)

resumo = {
    "modelo_selecionado": (
        final_nome
    ),
    "peso": str(BEST_FINAL),
    "atingiu_meta": atingiu_meta,
    "metas": {
        "mAP50": CFG.meta_map50,
        "precisao": (
            CFG.meta_precisao
        ),
        "recall": CFG.meta_recall,
        "mAP50_por_classe": (
            CFG.meta_map50_classe
        ),
    },
    "resultado": {
        chave: (
            float(valor)
            if isinstance(
                valor,
                (
                    np.floating,
                    float,
                    int,
                ),
            )
            else valor
        )
        for chave, valor
        in linha_final.to_dict().items()
    },
}

(
    RESULTADOS
    / "resumo_v4.json"
).write_text(
    json.dumps(
        resumo,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print(
    "Selecionado:",
    final_nome,
)
print(
    "Peso:",
    BEST_FINAL,
)
print(
    "Meta de 90% atingida:",
    atingiu_meta,
)

In [ ]:
# ============================================================
# PACOTE FINAL — modelo e métricas
# A exportação ONNX/TFLite fica para uma sessão limpa.
# ============================================================

if ENTREGA.exists():
    shutil.rmtree(
        ENTREGA
    )

ENTREGA.mkdir(
    parents=True,
    exist_ok=True,
)

shutil.copy2(
    BEST_FINAL,
    ENTREGA
    / "best_v4_selecionado.pt",
)

for arquivo in (
    RESULTADOS
    / "comparacao_modelos_v4.csv",
    RESULTADOS
    / "metricas_por_classe_v4.csv",
    RESULTADOS
    / "resumo_v4.json",
    yaml_v3,
):
    if arquivo.exists():
        shutil.copy2(
            arquivo,
            ENTREGA / arquivo.name,
        )

for pasta_resultado in (
    RESULTADOS
    / "weapons_v4_nano",
    RESULTADOS
    / "weapons_v4_small",
    RESULTADOS
    / "weapons_v4_refino",
):
    if not pasta_resultado.exists():
        continue

    destino = (
        ENTREGA
        / pasta_resultado.name
    )

    destino.mkdir(
        parents=True,
        exist_ok=True,
    )

    for nome in (
        "results.csv",
        "results.png",
        "confusion_matrix.png",
        "confusion_matrix_normalized.png",
        "PR_curve.png",
        "F1_curve.png",
    ):
        origem = (
            pasta_resultado / nome
        )

        if origem.exists():
            shutil.copy2(
                origem,
                destino / nome,
            )

ZIP_FINAL = shutil.make_archive(
    "/content/armas_v4_meta90_entrega",
    "zip",
    root_dir=str(ENTREGA),
)

print(
    "Pacote:",
    ZIP_FINAL,
)
print(
    "Modelo:",
    ENTREGA
    / "best_v4_selecionado.pt",
)

files.download(
    ZIP_FINAL
)